# Cortex Analyst — Natural Language Sales Q&A

**Cortex Analyst** converts natural-language questions into SQL using a **semantic model**.

| Item | Detail |
|---|---|
| **Capability** | Text-to-SQL via semantic model |
| **Data** | `SNOWFLAKE_SAMPLE_DATA.TPCH_SF1` — ORDERS, LINEITEM, CUSTOMER |
| **Architecture** | Question → Cortex Analyst API → LLM + semantic model → SQL → results |

### How it works
```
User Question → Cortex Analyst API → reads semantic model (YAML)
                                   → LLM generates SQL
                                   → SQL Result Set
```


## Step 1 — Environment & Views Setup

In [ ]:
CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

In [ ]:
CREATE OR REPLACE VIEW orders_vw AS
SELECT
    o_orderkey     AS order_key,
    o_custkey      AS customer_key,
    o_orderstatus  AS order_status,
    o_totalprice   AS total_price,
    o_orderdate    AS order_date,
    o_orderpriority AS order_priority,
    o_clerk        AS clerk,
    o_shippriority AS ship_priority,
    o_comment      AS order_comment
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS;

CREATE OR REPLACE VIEW lineitem_vw AS
SELECT
    l_orderkey      AS order_key,
    l_partkey       AS part_key,
    l_suppkey       AS supplier_key,
    l_linenumber    AS line_number,
    l_quantity      AS quantity,
    l_extendedprice AS extended_price,
    l_discount      AS discount,
    l_tax           AS tax,
    l_returnflag    AS return_flag,
    l_linestatus    AS line_status,
    l_shipdate      AS ship_date,
    l_commitdate    AS commit_date,
    l_receiptdate   AS receipt_date,
    l_shipinstruct  AS ship_instruct,
    l_shipmode      AS ship_mode,
    l_comment       AS line_comment
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.LINEITEM;

CREATE OR REPLACE VIEW customer_vw AS
SELECT
    c_custkey    AS customer_key,
    c_name       AS customer_name,
    c_address    AS address,
    c_nationkey  AS nation_key,
    c_phone      AS phone,
    c_acctbal    AS account_balance,
    c_mktsegment AS market_segment,
    c_comment    AS customer_comment
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER;

## Step 2 — Stage the Semantic Model

The semantic model YAML file (`sales_semantic_model.yaml`) must be uploaded to a stage.
In Snowflake Notebook, use the PUT command — the file should be in the same workspace.

In [ ]:
CREATE STAGE IF NOT EXISTS semantic_models
    COMMENT = 'Semantic model YAML files for Cortex Analyst';

In [ ]:
PUT file://structured/cortex-analyst-sales/sales_semantic_model.yaml
    @GENAI_LEARNING.PUBLIC.semantic_models
    OVERWRITE=TRUE
    AUTO_COMPRESS=FALSE;

## Step 3 — Data Exploration

In [ ]:
SELECT * FROM orders_vw LIMIT 5;

In [ ]:
SELECT
    (SELECT COUNT(*) FROM orders_vw)   AS orders_count,
    (SELECT COUNT(*) FROM lineitem_vw) AS lineitem_count,
    (SELECT COUNT(*) FROM customer_vw) AS customer_count;

## Step 4 — Example Queries (what Cortex Analyst generates)

Cortex Analyst is accessed via the **Snowsight AI assistant** or the **REST API**.
Below are example queries the semantic model's `verified_queries` would produce.

### "What was total revenue by month in 1995?"

In [ ]:
SELECT
    DATE_TRUNC('month', order_date) AS month,
    SUM(total_price)                AS total_revenue
FROM orders_vw
WHERE YEAR(order_date) = 1995
GROUP BY month
ORDER BY month;

### "Revenue by market segment?"

In [ ]:
SELECT
    c.market_segment,
    SUM(o.total_price) AS total_revenue
FROM orders_vw o
JOIN customer_vw c ON o.customer_key = c.customer_key
GROUP BY c.market_segment
ORDER BY total_revenue DESC;

### "Top 5 customers by total spend?"

In [ ]:
SELECT
    c.customer_name,
    c.market_segment,
    SUM(o.total_price) AS total_spend
FROM orders_vw o
JOIN customer_vw c ON o.customer_key = c.customer_key
GROUP BY c.customer_name, c.market_segment
ORDER BY total_spend DESC
LIMIT 5;

## Key Takeaways

- The **semantic model** defines business logic, column descriptions, and relationships.
- **`verified_queries`** are gold-standard question-SQL pairs that improve accuracy.
- Start with 5-10 verified queries covering your most common business questions.
- For production, consider **semantic views** (DDL objects) over stage-based YAML.
- Test with the Snowsight AI assistant to iterate on model quality.
